In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
#load 
sample_dt=pd.read_csv('/kaggle/input/ieee-fraud-detection/sample_submission.csv')
test_dt=pd.read_csv('/kaggle/input/ieee-fraud-detection/test_identity.csv')
test_transaction=pd.read_csv('/kaggle/input/ieee-fraud-detection/test_transaction.csv')
train_dt=pd.read_csv('/kaggle/input/ieee-fraud-detection/train_identity.csv')
train_transaction=pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')

In [ ]:
#understanding data
print(train_dt.shape)
print('\n')
print(train_dt.head())
print('\n')
print(train_dt.sample)
print('\n')
print(train_dt.info())
print('\n')
missing_value=train_dt.isnull().sum()
print(missing_value)
print(missing_value[missing_value>0])
print('\n')
print(train_dt.describe())
print('\n')
print(train_dt.duplicated().sum())
print('\n')
print(train_dt.corr(numeric_only=True))

In [ ]:
#understanding data
print(train_transaction.shape)
print('\n')
print(train_transaction.head())
print('\n')
print(train_transaction.sample)
print('\n')
print(train_transaction.info())
print('\n')
missing_value=train_transaction.isnull().sum()
print(missing_value)
print("train_transcation",missing_value[missing_value>0])
print('\n')
print(train_transaction.describe())
print('\n')
print(train_transaction.duplicated().sum())
print('\n')
print(train_transaction.corr(numeric_only=True))

In [ ]:
train=pd.merge(train_transaction,train_dt,on='TransactionID',how='left')
test=pd.merge(test_transaction,test_dt,on='TransactionID',how='left')


In [ ]:
print(train.shape)
print(train.head())

In [ ]:
train_Missing_Value=train.isnull().sum()
print(train_Missing_Value[train_Missing_Value>0])

In [ ]:
#calculating missing value percentage 
missing_value_percentage=train.isnull().sum()/len(train)*100
# colms with missing values >50%
colm_to_drop=missing_value_percentage[missing_value_percentage>50].index
#droping those colms
train_clean=train.drop(columns=colm_to_drop)
#droping t_id
train_clean=train_clean.drop(columns=['TransactionID','TransactionID'])
print("original colmns ", train.shape)
print("rows and columns left after droping ",train_clean.shape)
print("columns left after droping ",train_clean.shape[1])


In [ ]:
# only catgl colms
cat_cols=train_clean.select_dtypes(include=['object']).columns
print(train_clean[cat_cols].nunique().sort_values(ascending=False))

In [ ]:
# # fraud and not fruad cases
fraud_count=train_clean['isFraud'].value_counts()
fraud_percentage=train_clean['isFraud'].value_counts(normalize=True)*100
print("fraud counts :",fraud_count)
print("fraud percentage:",fraud_percentage)
sns.countplot(x='isFraud',data=train_clean)
plt.title('distribution of fraud (0=legit,1=fraud')
plt.show()

In [ ]:
#TransactionAmt vs Fraud
plt.figure(figsize=(12,6))
sns.boxplot(x='isFraud',y='TransactionAmt',data=train_clean)
plt.title('transaction amount distribution by fruad')
plt.ylim(0,1000)
plt.show()
#mean and median amt calculations 
print(train_clean.groupby('isFraud')['TransactionAmt'].agg(['mean','median','std']))


In [ ]:
# Create a table showing Fraud % for each Product Code
product_fraud=pd.crosstab(train_clean['ProductCD'],train_clean['isFraud'],normalize='index')*100
product_fraud.plot(kind='bar',figsize=(10,6),stacked=True)
plt.title('percntage of fraud by product code')
plt.ylabel('percentage')
plt.legend(title='isFraud',labels=['legit','Fraud'])
plt.show()
print(product_fraud)
#We found the "red flags" in the products; now we use Log Transform to turn messy transaction numbers into a nice "bell shape" so our model can learn easily.


In [ ]:
plt.figure(figsize=(12,5))
# Subplot 1: Original Distribution
plt.subplot(1,2,1)
sns.histplot(train_clean['TransactionAmt'],kde=True)
plt.title('original transaction ')
# Subplot 2: Log Transformed Distribution
# We use np.log1p (Log of 1 + x) to avoid errors if there are 0s
plt.subplot(1,2,2)
sns.histplot(np.log1p(train_clean['TransactionAmt']),kde=True,color='orange')
plt.title('log tranformed transaction amt ')
plt.show()

In [ ]:
# Find the 99th percentile value (Day 33)
cap_value = train_clean['TransactionAmt'].quantile(0.99)
print(f"The 99th percentile is: {cap_value}")

# Check how many transactions are above this cap
outliers = train_clean[train_clean['TransactionAmt'] > cap_value]
print(f"Number of transactions above the cap: {len(outliers)}")

In [ ]:
#We stop extreme numbers from "bullying" the model and reshape the data so the math works perfectly.
# 1. Cap the outliers at the 99th percentile (Day 33)
train_clean['TransactionAmt'] = train_clean['TransactionAmt'].clip(upper=cap_value)

# 2. Apply Log Transformation (Day 34)
train_clean['TransactionAmt'] = np.log1p(train_clean['TransactionAmt'])

print("Outliers capped and Log Transformation applied!")

In [ ]:
from sklearn.model_selection import train_test_split
# Define Features (X) and Target (y)
X = train_clean.drop(columns=['isFraud'])
y = train_clean['isFraud']

# Split the data (80% for training, 20% for testing our model)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify which columns are numbers and which are text
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['object']).columns

# 1. Numerical Pipeline (Impute + Scale)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Categorical Pipeline (Impute + Encode)
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

# 3. Combine them into one "Preprocessor" (Day 24)
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

print("The Cleaning Pipeline is ready!")

In [ ]:
from sklearn.decomposition import PCA

# 1. We create a new pipeline that first cleans the data, THEN applies PCA
# n_components=50 means we want to squash everything into the 50 most important features.
# (Later you can try 0.95 to keep 95% of the information)

full_pipeline = Pipeline([
    ('preprocessor', preprocessor), # The Imputer/Scaler/OHE factory we built
    ('pca', PCA(n_components=50))   # The Compressor (Day 42)
])

# 2. Let's see it in action on our training data
# Note: We use fit_transform on train, but only transform on validation (Day 7 logic)
print("Starting Preprocessing and PCA compression... This may take a few minutes.")
X_train_final = full_pipeline.fit_transform(X_train)
X_val_final = full_pipeline.transform(X_val)

print("Preprocessing and PCA complete!")
print(f"Original feature count (after OHE): Very high")
print(f"Compressed feature count: {X_train_final.shape[1]}")

In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. Initialize the Model (Day 45/54 logic)
# 'saga' is fast, 'l2' is Ridge regularization to prevent overfitting
log_model = LogisticRegression(solver='saga', penalty='l2', C=0.1, max_iter=100)

# 2. THE TRAINING (This is where the learning happens!)
print("Training started... The model is learning from 590k rows.")
log_model.fit(X_train_final, y_train) 
print("Model is trained! The brain is ready.")

In [ ]:
import gc
import numpy as np
import pandas as pd

# 1. Load the Test files
print("Loading test files...")
test_trans = pd.read_csv('/kaggle/input/ieee-fraud-detection/test_transaction.csv')
test_id = pd.read_csv('/kaggle/input/ieee-fraud-detection/test_identity.csv')

# 2. Merge them (Day 10 logic)
test = pd.merge(test_trans, test_id, on='TransactionID', how='left')

# 3. Save the TransactionID for later
test_ids = test['TransactionID']

# 4. Rename hyphens to underscores (Fixing the "Trap")
# This changes 'id-01' to 'id_01'
test.columns = [col.replace('-', '_') if 'id' in col else col for col in test.columns]

# 5. Drop the empty columns (Using the CORRECT variable name: columns_to_drop)
test_clean = test.drop(columns=colm_to_drop, errors='ignore')

# 6. Drop ONLY TransactionID
# WE DO NOT DROP TransactionDT HERE because your pipeline expects it!
test_clean = test_clean.drop(columns=['TransactionID'], errors='ignore')

# 7. Apply Outlier Capping and Log Transform (Day 33-34)
# We use the cap value 1104.0 we found during training
test_clean['TransactionAmt'] = test_clean['TransactionAmt'].clip(upper=1104.0)
test_clean['TransactionAmt'] = np.log1p(test_clean['TransactionAmt'])

# 8. Transform the test data using the Pipeline (The Assembly Line)
print("Transforming test data via Pipeline...")
# This applies Imputer, Scaler, OHE, and PCA automatically
X_test_final = full_pipeline.transform(test_clean)

# 9. Predict probabilities using the trained model (The Brain)
print("Generating final predictions...")
final_predictions = log_model.predict_proba(X_test_final)[:, 1]

# 10. Format and Save for Kaggle
submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud': final_predictions
})
submission.to_csv('submission.csv', index=False)

# 11. Free up memory
del test_trans, test_id, test, test_clean
gc.collect()

print("ALL DONE! Your file 'submission.csv' is ready for upload.")